In [1]:
import getpass
import os
from dotenv import load_dotenv

load_dotenv('../.env')
openai_api_key = os.getenv('OPENAI_ACCESS_KEY')
huggingface_access_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [2]:
from langchain_elasticsearch import ElasticsearchStore
from langchain_community.embeddings import HuggingFaceEmbeddings

from typing import Any, Dict, Iterable

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from langchain.embeddings import DeterministicFakeEmbedding
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_elasticsearch import ElasticsearchRetriever

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="dmlls/all-mpnet-base-v2-negation")
vector_dim = 768

/home/mabdallah/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mabdallah/miniconda3/lib/python3.11/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [4]:
# Your Elasticsearch Cloud endpoint
es_url = "https://4616afc5fbda42dfa82407bdaf369e18.us-central1.gcp.cloud.es.io"

# Basic authentication details
username = "elastic"
password = "gWQzXpyXFfQp9q3tWgBwCGNG"

# Create an Elasticsearch client with API Key authentication
es_client = Elasticsearch(
    hosts=[es_url],
    http_auth=(username, password),
)


# Test the connection
info = es_client.info()
print(info)


/tmp/ipykernel_3200346/1981531471.py:9: DeprecationWarning: The 'http_auth' parameter is deprecated. Use 'basic_auth' or 'bearer_auth' parameters instead
  es_client = Elasticsearch(


{'name': 'instance-0000000000', 'cluster_name': '4616afc5fbda42dfa82407bdaf369e18', 'cluster_uuid': 'M1TmpjezR3-smwcaMQr3ng', 'version': {'number': '8.13.4', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'da95df118650b55a500dcc181889ac35c6d8da7c', 'build_date': '2024-05-06T22:04:45.107454559Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [5]:
from elasticsearch import Elasticsearch

def create_index(es_client: Elasticsearch, index_name: str, vector_dims: int):
    es_client.indices.create(
        index=index_name,
        body={
            "settings": {
                "analysis": {
                    "analyzer": {
                        "standard_lowercase": {
                            "type": "custom",
                            "tokenizer": "standard",
                            "filter": ["lowercase"]
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "criterion": {
                        "type": "text",
                        "analyzer": "standard_lowercase"
                    },
                    "criterion_vector": {  # Renamed and moved to be a separate top-level field
                        "type": "dense_vector",
                        "dims": vector_dims
                    },
                    "entities": {
                        "type": "nested",
                        "properties": {
                            "normalized_id": {"type": "keyword"},
                            "synonyms": {"type": "text", "analyzer": "standard_lowercase"},
                            "is_negated": {"type": "keyword"},
                            "entity": {"type": "text", "analyzer": "standard_lowercase"},
                            "class": {"type": "keyword"}
                        }
                    },
                    "metadata": {
                        "properties": {
                            "nct_id": {"type": "keyword", "index": False}  # Stored but not indexed
                        }
                    }
                }
            }
        },
    )


In [6]:
from elasticsearch import Elasticsearch, NotFoundError, RequestError
def index_data(es_client, index_name, embeddings, documents, vector_dims, refresh=True):
    # Delete the index if it exists
    if es_client.indices.exists(index=index_name):
        es_client.indices.delete(index=index_name)
        print(f"Deleted existing index {index_name}")

    # Create the index
    create_index(es_client, index_name, vector_dims)

    indexed_count = 0  # To count how many documents have been indexed successfully
    for i, doc in enumerate(documents):
        vector = embeddings.embed_document(doc['criterion'])  # Generate embedding for the criterion
        document = {
            "criterion": doc['criterion'],
            "criterion_vector": vector,
            "entities": doc['entities'],
            "metadata": doc['metadata']
        }
        try:
            es_client.index(index=index_name, id=i, body=document)
            indexed_count += 1
        except NotFoundError as e:
            print(f"Index {index_name} not found: {e}")
        except RequestError as e:
            print(f"Request error during indexing document {i}: {e}")
        except Exception as e:
            print(f"Failed to index document {i}: {e}")

    if refresh:
        try:
            es_client.indices.refresh(index=index_name)
        except Exception as e:
            print(f"Failed to refresh index {index_name}: {e}")

    print(f"Successfully indexed {indexed_count} out of {len(documents)} documents.")
    return indexed_count

In [7]:
import os
import json

def prepare_documents(folder_path):
    documents = []
    # Loop through all files in the specified directory
    for filename in os.listdir(folder_path):
        if filename.endswith('.json'):  # Check for JSON files
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r') as file:
                data = json.load(file)  # Load JSON content
                nct_id = data['nct_id']  # Extract the NCT ID
                for criterion in data['criteria']:  # Iterate through each criterion
                    # Create a document for Elasticsearch
                    document = {
                        'criterion': criterion['criterion'],
                        'entities': criterion['entities'],
                        'type': criterion['type'],
                    }
                    documents.append(document)
    return documents

# Example usage
folder_path = '../../data/ner_trial/'
documents = prepare_documents(folder_path)
print(f"Prepared {len(documents)} documents for indexing.")


Prepared 47057 documents for indexing.


In [8]:
# Apply the function
number_of_indexed_documents = index_data(
    es_client=es_client,
    index_name="clinicaltrials",
    embeddings=embeddings,
    documents=documents,
    vector_dims=vector_dim
)

Deleted existing index clinicaltrials


BadRequestError: BadRequestError(400, 'mapper_parsing_exception', 'Failed to parse mapping: analyzer [standard_lowercase] has not been configured in mappings')

In [ ]:
def count_documents(es_client, index_name):
    try:
        count = es_client.count(index=index_name)
        print(f"Total documents in the index '{index_name}': {count['count']}")
        return count['count']
    except Exception as e:
        print(f"Error counting documents in index '{index_name}': {str(e)}")
        return None


In [ ]:
count_documents(es_client, "clinicaltrials")